# Yahoo XLNX 历史图表来源预检

固定一次只读 GET；验证该 URL 的历史价格覆盖，不读取 SOXX 结果。


In [1]:
import sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_xlnx_yahoo_history_preflight_20261023';snap=root/'data/snapshots/stage1_xlnx_yahoo_history_preflight_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((card/'plan.json').read_text());t=json.loads((card/'transport_result.json').read_text());a=t['attempt']
assert plan['frozen'] and t['plan_sha256']==sha(card/'plan.json') and t['GETs_consumed']==1 and t['retries']==0
assert sha(root/plan['prior_source_gate'])==plan['prior_source_gate_sha256'] and sha(root/plan['Polygon_credential_blocker'])==plan['Polygon_blocker_sha256']
assert a['method']==plan['exact_request']['method'] and a['path']==plan['exact_request']['path'] and a['query']==plan['exact_request']['query']
assert a['body_sha256']==sha(snap/'chart.json') and a['body_bytes']==(snap/'chart.json').stat().st_size
raw=json.loads((snap/'chart.json').read_text());assert a['http_status']==404 and raw['chart']['result'] is None and raw['chart']['error']['code']=='Not Found'
assert 'delisted' in raw['chart']['error']['description']
report={'decision':'fixed_Yahoo_XLNX_chart_URL_not_found_no_historical_bars','plan_sha256':sha(card/'plan.json'),'HTTP_status':404,'provider_error':raw['chart']['error'],'historical_daily_price_rows':0,'2019_2022_constituent_price_source_qualified':False,'historical_first_receipt_qualified':False,'SOXX_outcomes_read':0,'new_model_fits':0,'new_native_accounts':0,'GETs':1,'retries':0}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
settings=_settings(root);members=[card/'plan.json',card/'transport_result.json',snap/'chart.json']
with tempfile.TemporaryDirectory(prefix='xlnx-yahoo-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in members:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive)
 response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')
 receipt=response['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in members}
  for p in members:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
receipt_out={'status':'source_archive_NAS_scoped_restore_verified','sha256':digest,'size_bytes':receipt['size_bytes'],'nas_path':nas_path,'members':{str(p.relative_to(root)):sha(p) for p in members},'verified':True}
(card/'nas_source_archive.json').write_text(json.dumps(receipt_out,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','model_fits':0,'native_accounts':0,'market_outcome_reads':0,'Qlib_Recorder':None,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]},'limitations':['The 404 closes only this exact Yahoo chart URL and requested ticker/date range.','A delisted-ticker error is not proof XLNX history does not exist elsewhere.','No historical ticker prices, factor, or SOXX account result.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'HTTP_status':404,'error':raw['chart']['error']['description'],'nas_sha256':digest,'fits':0,'accounts':0}))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"HTTP_status": 404, "error": "No data found, symbol may be delisted", "nas_sha256": "b9859756d7a735d42cdb17e2984ca32444459687f04d8be9ac08aa34af95cd08", "fits": 0, "accounts": 0}


# Yahoo XLNX 历史图表：固定 URL 来源预检

按冻结计划，只请求一次 `XLNX` 从 2019-01-01 起、以 2022-12-31 00:00 UTC 为不含结束上界的 Yahoo 日线图表。HTTP 404；响应为 `Not Found`，提示 `No data found, symbol may be delisted`，历史价格行数为 0。原始响应、请求参数和冻结计划已按 SHA-256 归档 Yanxuan NAS，并完成远端限定恢复。

这只关闭**该 URL、该符号和该日期范围**，不证明退市 XLNX 的价格在其他数据源不可得，也不能把收购方 AMD 价格拼接过去。此前 Polygon 当前凭证的相同历史范围被拒绝授权；两条入口都没有为 SOXX 已披露 30 股篮子补齐 XLNX 的 630 个缺价成员日。

0 SOXX 收益读取、0 模型拟合、0 账户。**决定：不形成下跌集中度特征，不推广。** 下一步仅在可验证的原证券历史价格和公司行动来源出现时继续这条来源支线；其它合格机制可并行研究。
